# Building a RAG system with LangChain and ChromaDB

We are going to use:
1. LangChain: a framework for developing applications powered by language models
2. ChromaDB: an open-source vector databases for storing and retrieving embeddings
3. OpenAI: for embeddings and language model

In [1]:
import os
from dotenv import load_dotenv

load_dotenv
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [2]:
# langchain imports
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# vector stores import
from langchain_community.vectorstores import Chroma 

# utility imports
import numpy as np
from typing import List

c:\code2\Natural Language Processing Projects\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: load documents from various soures

2. Document Splitting: break documents into smaller chunks 

3. Embedding Generation: convert chunks into vector representations

4. Vector Storage: store embedding vectors in ChromaDB

5. Query Processing: convert user query to embedding vector

6. Similarity Search: find relevant chunks from vector store

7. Context Augmentation: combine retrieved chunks with query

8. Response Generation: LLM generates answer using context

# Create sample documents

In [3]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [4]:
# save documents to files
import tempfile

temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc{i}.txt","w") as f:
        f.write(doc)

In [5]:
print(f"Sample document create in: {temp_dir}")

Sample document create in: C:\Users\PHUCTH~1\AppData\Local\Temp\tmphtr1q7nh


# Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load documents from directory
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)

documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
for i, doc in enumerate(documents):
    print(f"Document {i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")
    print()

100%|██████████| 3/3 [00:00<00:00, 652.91it/s]

Loaded 3 document(s)
Document 1:
Metadata: {'source': 'C:\\Users\\PHUCTH~1\\AppData\\Local\\Temp\\tmphtr1q7nh\\doc0.txt'}
Content: 
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    

Document 2:
Metadata: {'source': 'C:\\Users\\PHUCTH~1\\AppData\\Local\\Temp\\tmphtr1q7nh\\doc1.txt'}
Content: 
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers 

# Document Splitting

In [7]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(chunks)} chunk(s)")
print("Example chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Loaded 5 chunk(s)
Example chunk:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through
Metadata: {'source': 'C:\\Users\\PHUCTH~1\\AppData\\Local\\Temp\\tmphtr1q7nh\\doc0.txt'}


# Initialize ChromaDB vector store and store chunks in vector representation

In [8]:
# Loading model
embeddings = OpenAIEmbeddings()

In [9]:
persist_directory = './chroma_db'

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {vector_store._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 5 vectors
Persisted to: ./chroma_db


# Similarity search

In [10]:
query = 'Tell me about NLP'

similar_docs = vector_store.similarity_search(query, k=1)
similar_docs

[Document(metadata={'source': 'C:\\Users\\PHUCTH~1\\AppData\\Local\\Temp\\tmphtr1q7nh\\doc2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.')]

In [11]:
# similarity_search_with_score

similar_docs = vector_store.similarity_search_with_score(query, k=3)
similar_docs

[(Document(metadata={'source': 'C:\\Users\\PHUCTH~1\\AppData\\Local\\Temp\\tmphtr1q7nh\\doc2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.23224228620529175),
 (Document(metadata={'source': 'C:\\Users\\PHUCTH~1\\AppData\\Local\\Temp\\tmphtr1q7nh\\doc1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields li

# Initialize LLM

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name='gpt-3.5-turbo'
)

In [13]:
test_response = llm.invoke("What is LLM")
print(test_response)

content='LLM stands for Master of Laws, which is an advanced postgraduate law degree typically undertaken by individuals who already possess a professional law degree. It is designed to provide specialized knowledge and expertise in a specific area of law.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 11, 'total_tokens': 55, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CUPFFsqrez8sXt9APIEllRSL9Db3u', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--4759ba51-4803-47b4-b124-4380d8d8bc44-0' usage_metadata={'input_tokens': 11, 'output_tokens': 44, 'total_tokens': 55, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

In [14]:
# Another way to call a model
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-3.5-turbo")

# RAG pipeline with LCEL

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [16]:
retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)

In [17]:
prompt = ChatPromptTemplate.from_template(
"""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}
Question: {question}
"""
)


In [18]:
rag_chain = (
    {"context":retriever, "question":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
response = rag_chain.invoke("tell me about AI")
print(response)

Artificial intelligence (AI) encompasses subsets like machine learning, deep learning, and natural language processing. Machine learning allows systems to learn and improve from experience without explicit programming, including supervised, unsupervised, and reinforcement learning. Deep learning, inspired by the human brain, uses artificial neural networks to revolutionize fields like computer vision and natural language processing.


# Add new documents to vector store

In [20]:
# Create new document
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [21]:
# turn into langchain document datatype
new_doc = Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [22]:
# split document into chunks
new_chunks = text_splitter.split_documents([new_doc])

print(f"Splitted into {len(new_chunks)} chunk(s)")
print(f"First chunk content: {new_chunks[0].page_content}")
print(f"First chunk metadata: {new_chunks[0].metadata}")

Splitted into 2 chunk(s)
First chunk content: Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been
First chunk metadata: {'source': 'manual_addition', 'topic': 'reinforcement_learning'}


In [23]:
# Add new documents to vector store
vector_store.add_documents(new_chunks)

['792c3644-cfb7-4bc9-91db-6147c7e3d6d8',
 'ad19e50a-c189-415d-bb78-3fbadfff8165']

In [24]:
print(f"Total vectors in vector store now: {vector_store._collection.count()}")

Total vectors in vector store now: 7


In [27]:
new_question="What is RL stand for"
response = rag_chain.invoke(new_question)
print(response)

RL stands for Reinforcement Learning.


In [ ]:
from langchain_classic.chains import create_history_aware_retriever

ModuleNotFoundError: No module named 'langchain.chains'